In [31]:
#imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pyarrow

In [32]:
#data
transactions = pd.read_csv("../data/transactions_2016_2017.csv")
train = pd.read_csv("../data/customer_clv_train.csv")
test = pd.read_csv("../data/customer_clv_test.csv")

/tmp/ipykernel_14737/2271748159.py:2: DtypeWarning: Columns (0: prod_size) have mixed types. Specify dtype option on import or set low_memory=False.
  transactions = pd.read_csv("../data/transactions_2016_2017.csv")


In [33]:
transactions["order_date"] = pd.to_datetime(transactions["order_date"])
transactions["pack_date"] = pd.to_datetime(transactions["pack_date"])

# RFM Features

In [34]:
reference_date = transactions["order_date"].max()

In [35]:
# identify returns and purchases
transactions["is_return"] = transactions["sale_revenue"] < 0
transactions["is_purchase"] = transactions["sale_revenue"] > 0

transactions["discounted_item"] = transactions["sale_discount_applied"] < 0

transactions["positive_revenue"] = transactions["sale_revenue"].clip(lower=0)
transactions["returned_revenue_abs"] = transactions["sale_revenue"].clip(upper=0).abs()

In [36]:
# create customer features
customer_features = (
    transactions
    .groupby("cust_id")
    .agg(
        # Monetary
        total_revenue=("sale_revenue", "sum"),
        max_item_revenue=("sale_revenue", "max"),

        n_items=("sale_id", "count"),

        # Frequency
        n_orders=("sale_id", "nunique"),

        # Discounts
        total_discount=("sale_discount_applied", "sum"),

        # Product diversity
        n_products=("prod_id", "nunique"),
        n_brands=("prod_brand", "nunique"),
        n_colors=("prod_color", "nunique"),
        n_categories=("prod_type_3", "nunique"),

        # Size behaviour
        avg_size=("prod_size", lambda x: pd.to_numeric(x, errors="coerce").mean()),
        size_std=("prod_size", lambda x: pd.to_numeric(x, errors="coerce").std()),
        n_sizes=("prod_size", "nunique"),

        # Dates
        first_purchase=("order_date", "min"),
        last_purchase=("order_date", "max")
    )
    .reset_index()
)

In [37]:
customer_features["recency_days"] = (
    reference_date - customer_features["last_purchase"]
).dt.days

customer_features["customer_age_days"] = (
    customer_features["last_purchase"] - customer_features["first_purchase"]
).dt.days

customer_features = customer_features.drop(
    columns=["first_purchase", "last_purchase"]
)

In [38]:
customer_features["size_std"] = customer_features["size_std"].fillna(0)

In [39]:
# intensity features
customer_features["orders_per_day"] = (
    customer_features["n_orders"] /
    (customer_features["customer_age_days"] + 1)
)

customer_features["frequency_recency_ratio"] = (
    customer_features["n_orders"] /
    (customer_features["recency_days"] + 1)
)

customer_features["revenue_recency_ratio"] = (
    customer_features["total_revenue"] /
    (customer_features["recency_days"] + 1)
)

customer_features["discount_per_item"] = (
    customer_features["total_discount"] /
    (customer_features["n_items"] + 1)
)

In [40]:
# Average Gaps + Purchase Irregularity

transactions_sorted = transactions.sort_values(["cust_id", "order_date"]).copy()

transactions_sorted["prev_order_date"] = (
    transactions_sorted.groupby("cust_id")["order_date"].shift()
)

transactions_sorted["days_between_orders"] = (
    transactions_sorted["order_date"] - transactions_sorted["prev_order_date"]
).dt.days

avg_gap = transactions_sorted.groupby("cust_id")["days_between_orders"].mean()
max_gap = transactions_sorted.groupby("cust_id")["days_between_orders"].max()

customer_features["avg_days_between_orders"] = customer_features["cust_id"].map(avg_gap)
customer_features["max_days_between_orders"] = customer_features["cust_id"].map(max_gap)

customer_features["avg_days_between_orders"] = (
    customer_features["avg_days_between_orders"].fillna(-999)
)
customer_features["max_days_between_orders"] = (
    customer_features["max_days_between_orders"].fillna(-999)
)

In [41]:
## ADDED BY FIEN 
# recency weighted features (recent transactions => larger weight)

# prep - creation days from ref and recency weight
transactions["days_from_ref"] = (reference_date - transactions["order_date"]).dt.days

DECAY_DAYS = 90                     # 90 is kind of standard value decay days in CLV 
                                    # BUT: could be tuned as hyperparameter
transactions["recency_weight"] = np.exp(-transactions["days_from_ref"] / DECAY_DAYS)

# weighted revenue 
order_level_revenue = (
    transactions
    .groupby(["cust_id", "sale_id"])
    .agg(
        order_revenue=("positive_revenue", "sum"),
        order_weight=("recency_weight", "mean")
    )
    .reset_index()
)

weighted_revenue = (
    order_level_revenue["order_revenue"] * order_level_revenue["order_weight"]
)

weighted_revenue = (
    order_level_revenue.assign(weighted_value=weighted_revenue)
    .groupby("cust_id")["weighted_value"]
    .sum()
)

# weighted orders
order_level = (
    transactions
    .groupby(["cust_id", "sale_id"])
    .agg(
        order_weight=("recency_weight", "mean")
    )
    .reset_index()
)

weighted_orders = (
    order_level
    .groupby("cust_id")["order_weight"]
    .sum()
)

# creation of features weighted revenue and weighted orders
customer_features["weighted_revenue"] = customer_features["cust_id"].map(weighted_revenue).fillna(0)
customer_features["weighted_orders"] = customer_features["cust_id"].map(weighted_orders).fillna(0)

# creation of feature weighted avg order value => TO BE KEPT? 
customer_features["weighted_avg_order_value"] = (
    customer_features["weighted_revenue"] /
    (customer_features["weighted_orders"] + 1)
)


In [42]:
all_disc = transactions.groupby("cust_id").agg(
    all_items_discounted=("discounted_item", "all"),
    all_orders_discounted=("discounted_item", lambda x:
        transactions.loc[x.index, "sale_id"][x].nunique() ==
        transactions.loc[x.index, "sale_id"].nunique())
).reset_index()
customer_features = customer_features.merge(all_disc, on="cust_id", how="left")

# Features added by me

In [43]:
# add delivery_days to transactions for avg_delivery_days feature
transactions["delivery_days"] = (transactions["pack_date"] - transactions["order_date"]).dt.days

# aggregate new features at customer level
new_features = (
    transactions
    .groupby("cust_id")
    .agg(
        web_only_rate=("prod_web_only", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        n_seasons=("prod_season", "nunique"),
    )
    .reset_index()
)

# merge into customer_features
customer_features = customer_features.merge(new_features, on="cust_id", how="left")


# FIEN FEATURES

In [44]:
# 6 Recency per categorie — meest recente aankoop per schoentype
last_cat_purchase = (
    transactions.groupby(["cust_id", "prod_type_3"])["order_date"]
    .max()
    .reset_index()
)
last_cat_purchase["days_since"] = (
    pd.Timestamp("2017-12-31") - last_cat_purchase["order_date"]
).dt.days

top_cat = (
    transactions.groupby("cust_id")["prod_type_3"]
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={"prod_type_3": "top_category"})
)
recency_top_cat = top_cat.merge(last_cat_purchase, 
    left_on=["cust_id", "top_category"],
    right_on=["cust_id", "prod_type_3"], how="left")

customer_features["recency_top_category"] = (
    customer_features["cust_id"].map(
        recency_top_cat.set_index("cust_id")["days_since"]
    ).fillna(-999)  # sentinel voor ontbrekende categorie info
)

In [45]:
# size_std sentinel -999 voor single-order klanten
customer_features["size_std"] = customer_features.apply(
    lambda r: -999 if r["n_orders"] == 1 else r["size_std"], axis=1
)

# Schoentype proporties
shoe_flags = {
    "is_boot":        lambda x: x.str.contains("boot", na=False),
    "is_sneaker":     lambda x: x.str.contains("sneaker", na=False),
    "is_sandal":      lambda x: x.str.contains("sandal|flip-flop", na=False),
    "is_high_shoe":   lambda x: x.str.contains("high shoe", na=False),
    "is_low_shoe":    lambda x: x.str.contains("low shoe", na=False),
    "is_pump":        lambda x: x.str.contains("pump", na=False),
    "is_slipper":     lambda x: x.str.contains("slipper", na=False),
}
for flag, func in shoe_flags.items():
    transactions[flag] = func(transactions["prod_type_3"]).astype(int)

shoe_pcts = transactions.groupby("cust_id").agg(
    pct_boot        =("is_boot",        "mean"),
    pct_sneaker     =("is_sneaker",     "mean"),
    pct_sandal      =("is_sandal",      "mean"),
    pct_high_shoe   =("is_high_shoe",   "mean"),
    pct_low_shoe    =("is_low_shoe",    "mean"),
    pct_pump        =("is_pump",        "mean"),
    pct_slipper     =("is_slipper",     "mean"),
).reset_index()
customer_features = customer_features.merge(shoe_pcts, on="cust_id", how="left")

# Segment flags
transactions["is_velcro"]       = transactions["prod_clasp"].str.contains("velcro", na=False).astype(int)
transactions["is_leather"]      = transactions["prod_material"].str.contains("leather", na=False).astype(int)

segment_flags = transactions.groupby("cust_id").agg(
    has_velcro      =("is_velcro",       "max"),
    has_leather     =("is_leather",      "max"),
).reset_index()
customer_features = customer_features.merge(segment_flags, on="cust_id", how="left")

In [46]:
## ADDED BY FIEN
# interaction features

customer_features["recency_x_frequency"] = (
    customer_features["recency_days"] * customer_features["n_orders"]
)

customer_features["size_std_x_orders"] = (
    customer_features["size_std"] * customer_features["n_orders"]
)

In [47]:
# Kevin's features: sales timing, gender segment, return shop
transactions["year"]           = transactions["order_date"].dt.year
transactions["is_sales_month"] = transactions["order_date"].dt.month.isin([1, 7])

gender_feats = (
    transactions.groupby("cust_id").agg(
        men_items_only     =("prod_type_1", lambda x: int((x == "men").all())),
    ).reset_index()
)
customer_features = customer_features.merge(gender_feats, on="cust_id", how="left")
for col in ["men_items_only"]:
    customer_features[col] = customer_features[col].fillna(0).astype(int)

print(f"Features toegevoegd. Totaal: {len(customer_features.columns) - 1}")

Features toegevoegd. Totaal: 41


In [48]:
# pct_orders_with_discount: fractie orders met korting
pct_disc = (
    transactions.groupby("cust_id")
    .apply(lambda x: x.loc[x["discounted_item"], "sale_id"].nunique() / x["sale_id"].nunique())
    .rename("pct_orders_with_discount")
    .reset_index()
)
customer_features = customer_features.merge(pct_disc, on="cust_id", how="left")
customer_features["pct_orders_with_discount"] = customer_features["pct_orders_with_discount"].fillna(0)

In [49]:
# === Batch 2 features ===

# 1. Seizoenaliteit: eerste en laatste aankoopmaand
fp_lp = transactions.groupby("cust_id")["order_date"].agg(["min", "max"])
customer_features["first_purchase_month"] = customer_features["cust_id"].map(fp_lp["min"].dt.month)

# 2. Gemiddelde revenue per item, exclusief returns
purchase_agg = (
    transactions[transactions["is_purchase"]]
    .groupby("cust_id")["sale_revenue"]
    .agg(["sum", "count"])
)

# Brand concentration (Herfindahl) — vectorized
brand_counts = (
    transactions.groupby(["cust_id", "prod_brand"])
    .size()
    .reset_index(name="n")
)
brand_totals = brand_counts.groupby("cust_id")["n"].transform("sum")
brand_counts["p"] = brand_counts["n"] / brand_totals
brand_conc = (
    (brand_counts["p"] ** 2)
    .groupby(brand_counts["cust_id"])
    .sum()
    .rename("brand_concentration")
)
customer_features["brand_concentration"] = customer_features["cust_id"].map(brand_conc).fillna(1)

# Category concentration (Herfindahl) — vectorized
cat_counts = (
    transactions.groupby(["cust_id", "prod_type_3"])
    .size()
    .reset_index(name="n")
)
cat_totals = cat_counts.groupby("cust_id")["n"].transform("sum")
cat_counts["p"] = cat_counts["n"] / cat_totals
cat_conc = (
    (cat_counts["p"] ** 2)
    .groupby(cat_counts["cust_id"])
    .sum()
    .rename("category_concentration")
)
customer_features["category_concentration"] = customer_features["cust_id"].map(cat_conc).fillna(1)

# 5. Relatieve kortingsgraad
customer_features["discount_rate"] = (
    customer_features["total_discount"] /
    (customer_features["total_revenue"] + customer_features["total_discount"] + 1e-6)
)

# 6. New-season share: aandeel aankopen van seizoen W16/S17 of recenter
import re

def parse_season_year(s):
    if pd.isna(s):
        return np.nan
    m = re.search(r'(\d+)', str(s))
    if m:
        yr = int(m.group(1))
        return 2000 + yr if yr < 100 else yr
    return np.nan

transactions["season_year"] = transactions["prod_season"].apply(parse_season_year)

# 8. Recency x revenue interactie
customer_features["recency_x_revenue"] = (
    customer_features["recency_days"] * customer_features["total_revenue"]
)

# 10. Brand en category return rates (product-level, geen target leakage)
purchases_only = transactions[transactions["is_purchase"]].copy()

primary_brand = (
    purchases_only.groupby("cust_id")["prod_brand"]
    .agg(lambda x: x.value_counts().index[0] if len(x) > 0 else "unknown")
    .reset_index()
    .rename(columns={"prod_brand": "primary_brand"})
)
primary_cat = (
    purchases_only.groupby("cust_id")["prod_type_3"]
    .agg(lambda x: x.value_counts().index[0] if len(x) > 0 else "unknown")
    .reset_index()
    .rename(columns={"prod_type_3": "primary_category"})
)
customer_features = (
    customer_features
    .merge(primary_brand, on="cust_id", how="left")
    .merge(primary_cat,   on="cust_id", how="left")
)
customer_features["primary_brand"]    = customer_features["primary_brand"].fillna("unknown")
customer_features["primary_category"] = customer_features["primary_category"].fillna("unknown")

global_return_rate = transactions["is_return"].mean()

brand_return_rate = (
    transactions.groupby("prod_brand")["is_return"].mean()
    .reset_index()
    .rename(columns={"prod_brand": "primary_brand", "is_return": "brand_return_rate"})
)
customer_features = (
    customer_features
    .merge(brand_return_rate, on="primary_brand",    how="left")
)
customer_features["brand_return_rate"]    = customer_features["brand_return_rate"].fillna(global_return_rate)

# Drop tussenproducten
customer_features = customer_features.drop(columns=["primary_brand", "primary_category"])

print(f"Batch 2 features toegevoegd. Totaal: {len(customer_features.columns) - 1}")

Batch 2 features toegevoegd. Totaal: 48


In [50]:
# batch 3
# 1. Revenue std — spread van orderwaarden
order_rev = (
    transactions[transactions["is_purchase"]]
    .groupby(["cust_id", "sale_id"])["sale_revenue"].sum()
)

rev_std = order_rev.groupby("cust_id").std()
customer_features["revenue_std"] = customer_features["cust_id"].map(rev_std)
customer_features["revenue_std"] = np.where(
    customer_features["n_orders"] == 1,
    -999,
    customer_features["revenue_std"].fillna(0)
)

# 3. Eerste aankoop in sale maand
first_order_month = transactions.groupby("cust_id")["order_date"].min().dt.month
customer_features["first_order_in_sales_month"] = (
    customer_features["cust_id"].map(first_order_month).isin([1, 7])
).astype(int)

# 4. Pct insole
trans_insole = transactions.copy()
trans_insole["has_insole"] = (trans_insole["prod_insole"] == 1)
pct_insole = trans_insole.groupby("cust_id")["has_insole"].mean()
customer_features["pct_insole"] = customer_features["cust_id"].map(pct_insole).fillna(0)

In [51]:
customer_features["order_segment"] = pd.cut(
    customer_features["n_orders"], 
    bins=[0, 1, 3, np.inf], 
    labels=[0, 1, 2]
).astype(int)

In [52]:
median_order_revenue = (
    transactions[transactions["is_purchase"]]
    .groupby(["cust_id", "sale_id"])["sale_revenue"]
    .sum()
    .groupby("cust_id")
    .median()
)
customer_features["median_order_revenue"] = customer_features["cust_id"].map(median_order_revenue).fillna(0)

In [53]:
top_cat_share = (
    transactions.groupby(["cust_id", "prod_type_3"])
    .size()
    .reset_index(name="n")
)
totals = top_cat_share.groupby("cust_id")["n"].sum()
top_cat_share["share"] = top_cat_share["n"] / top_cat_share["cust_id"].map(totals)
top_cat_pct = top_cat_share.groupby("cust_id")["share"].max()
customer_features["top_category_share"] = customer_features["cust_id"].map(top_cat_pct).fillna(1)

In [54]:
# === Kwartaalfeatures ===
# 6 kwartalen in de observatieperiode: 2016Q1 t/m 2017Q2
# Revenue, orders en returns per kwartaal per klant
# Geeft het model sequentiële info over gedragsevolutie

valid_quarters = ["2016Q2", "2016Q3", "2016Q4", "2017Q1", "2017Q2"]
transactions["quarter"] = transactions["order_date"].dt.to_period("Q").astype(str)
tx_q = transactions[transactions["quarter"].isin(valid_quarters)].copy()

# Revenue per kwartaal
q_revenue = (
    tx_q.groupby(["cust_id", "quarter"])["sale_revenue"]
    .sum()
    .unstack(fill_value=0)
    .add_prefix("q_rev_")
    .reindex(columns=[f"q_rev_{q}" for q in valid_quarters], fill_value=0)
)

customer_features = (
    customer_features
    .merge(q_revenue.reset_index(),  on="cust_id", how="left")
)

# Klanten zonder activiteit in een kwartaal krijgen 0
q_cols = [c for c in customer_features.columns 
          if c.startswith("q_rev_")]
for col in q_cols:
    customer_features[col] = customer_features[col].fillna(0)

print(f"Kwartaalfeatures toegevoegd: {len(q_cols)} features. Totaal: {len(customer_features.columns) - 1}")

Kwartaalfeatures toegevoegd: 5 features. Totaal: 59


In [55]:
# Leveringsconsistentie
delivery_stats = (
    transactions
    .groupby("cust_id")["delivery_days"]
    .agg(["std", "mean"])
    .rename(columns={"std": "delivery_days_std", "mean": "delivery_days_mean"})
)

# Eerst mappen
customer_features["delivery_days_std"] = customer_features["cust_id"].map(
    delivery_stats["delivery_days_std"]
)
customer_features["delivery_days_mean"] = customer_features["cust_id"].map(
    delivery_stats["delivery_days_mean"]
)

# Bereken ratio met echte waarden
customer_features["delivery_consistency"] = (
    customer_features["delivery_days_std"] /
    (customer_features["delivery_days_mean"] + 1)
)

# Dan pas sentinel voor beide
customer_features["delivery_days_std"] = np.where(
    customer_features["n_orders"] == 1,
    -999,
    customer_features["delivery_days_std"].fillna(0)
)
customer_features["delivery_consistency"] = np.where(
    customer_features["n_orders"] == 1,
    -999,
    customer_features["delivery_consistency"].fillna(0)
)
customer_features = customer_features.drop(columns=["delivery_days_mean"])

In [56]:
# first_order_gap: dagen tussen eerste en tweede aankoop
# signaal voor vroege loyaliteit
transactions_sorted2 = transactions.sort_values(["cust_id", "order_date"]).copy()
transactions_sorted2["prev_order_date"] = (
    transactions_sorted2.groupby("cust_id")["order_date"].shift()
)
transactions_sorted2["days_between"] = (
    transactions_sorted2["order_date"] - transactions_sorted2["prev_order_date"]
).dt.days

first_gap = (
    transactions_sorted2.dropna(subset=["days_between"])
    .groupby("cust_id")["days_between"]
    .first()
)
customer_features["first_order_gap"] = (
    customer_features["cust_id"].map(first_gap).fillna(-999)
)

print(f"Extra features toegevoegd. Totaal: {len(customer_features.columns) - 1}")

Extra features toegevoegd. Totaal: 62


In [57]:
# === Batch 4: RFM trend features ===

# 1. Gap trend: worden gaps groter of kleiner over tijd?
# Positief = gaps groeien = churnsignaal
# Sentinel -999 voor single-order klanten (geen gaps)
transactions_sorted3 = (
    transactions
    .drop_duplicates(subset=["cust_id", "sale_id"])
    .sort_values(["cust_id", "order_date"])
    .copy()
)
transactions_sorted3["prev_order_date"] = (
    transactions_sorted3.groupby("cust_id")["order_date"].shift()
)
transactions_sorted3["gap_days"] = (
    transactions_sorted3["order_date"] - transactions_sorted3["prev_order_date"]
).dt.days

first_gap_per_cust = (
    transactions_sorted3.dropna(subset=["gap_days"])
    .groupby("cust_id")["gap_days"].first()
)
last_gap_per_cust = (
    transactions_sorted3.dropna(subset=["gap_days"])
    .groupby("cust_id")["gap_days"].last()
)
gap_trend = last_gap_per_cust - first_gap_per_cust

customer_features["gap_trend"] = np.where(
    customer_features["n_orders"] <= 2,
    -999,  # sentinel: geen zinvolle trend met minder dan 3 orders
    customer_features["cust_id"].map(gap_trend).fillna(-999)
)

# 2. Revenue per lifetime day: CLV proxy
customer_features["revenue_per_lifetime_day"] = (
    customer_features["total_revenue"] /
    (customer_features["customer_age_days"] + 1)
)

# 3. Repeat purchase rate: koopt iemand hetzelfde product opnieuw? => feature weg maar dit nog nodig voor andere
# feature caculations
purchase_tx = transactions[transactions["is_purchase"]].copy()

# 4. Laatste order vs gemiddelde order: is klant minder gaan uitgeven?
# Veiligere versie — sorteren op order_date
last_order_rev = (
    purchase_tx
    .groupby(["cust_id", "sale_id", "order_date"])["sale_revenue"]
    .sum()
    .reset_index()
    .sort_values(["cust_id", "order_date"])
    .groupby("cust_id")["sale_revenue"]
    .last()
)
avg_order_rev = (
    purchase_tx
    .groupby(["cust_id", "sale_id"])["sale_revenue"]
    .sum()
    .groupby("cust_id")
    .mean()
)
customer_features["last_vs_avg_order"] = (
    customer_features["cust_id"].map(last_order_rev) /
    (customer_features["cust_id"].map(avg_order_rev) + 1e-6)
).fillna(1).clip(0, 5)

# 5. Purchase acceleration: kocht sneller in 2017 dan 2016?
# Sentinel -999 voor klanten die enkel in één jaar actief waren
ord_2016 = (
    transactions[transactions["year"] == 2016]
    .groupby("cust_id")["sale_id"].nunique()
)
ord_2017 = (
    transactions[transactions["year"] == 2017]
    .groupby("cust_id")["sale_id"].nunique()
)
years_active = transactions.groupby("cust_id")["year"].nunique()

customer_features["purchase_acceleration"] = np.where(
    customer_features["cust_id"].map(years_active) < 2,
    -999,
    customer_features["cust_id"].map(ord_2017).fillna(0) -
    customer_features["cust_id"].map(ord_2016).fillna(0)
)

print(f"Batch 4 features toegevoegd. Totaal: {len(customer_features.columns) - 1}")

Batch 4 features toegevoegd. Totaal: 66


In [58]:
# discount_x_frequency: prijsgevoeligheid × aankoopfrequentie
customer_features["discount_x_frequency"] = (
    customer_features["discount_rate"] * customer_features["n_orders"]
)

# max_gap_x_recency: sentinel voor single-order klanten
customer_features["max_gap_x_recency"] = np.where(
    customer_features["n_orders"] == 1,
    -999,
    customer_features["max_days_between_orders"] * customer_features["recency_days"]
)

In [59]:
customer_features = customer_features.copy()

In [60]:
customer_features.to_csv("../data/customer_features_lowie1.csv", index=False)